# Speaker Diarization with Pyannote on VAST

## Introduction

This notebook demonstrates how to implement Speaker Diarization using the Pyannote Audio library on VAST.ai's cloud computing platform. Speaker Diarization is the process of partitioning an audio stream into segments according to the speaker identity, answering the question "who spoke when?"

### Why Speaker Diarization Matters

Speaker Diarization provides several key benefits for audio processing pipelines:

1. **Speaker Identification**: It identifies different speakers in a conversation, meeting, or any multi-speaker audio recording.

2. **Improved Transcription**: When combined with speech-to-text systems, diarization allows for speaker-attributed transcripts, making it clear who said what.

3. **Processing Efficiency**: By segmenting audio by speaker and removing non-speech portions, diarization can significantly reduce the computational load for downstream tasks like speech recognition, allowing these systems to process only relevant speech segments rather than the entire audio file.

4. **Audio Indexing**: Makes audio content searchable by speaker, allowing users to find all segments where a specific person speaks.


### What This Notebook Does

In this notebook, we will:
- Set up the Pyannote Audio Speaker Diarization pipeline
- Process audio files to detect different speakers and their speaking turns
- Calculate speaking time for each identified speaker
- Identify regions with overlapping speech
- Extract and save speaker-specific segments from the input audio
- Play and verify the diarization results

The output will be a collection of audio files separated by speaker, making them ready for further processing in speech-to-text pipelines or speaker-specific analysis.

### Why VAST.ai

Vast is particularly well-suited for running speaker diarization models, as these typically require GPU acceleration but don't demand extensive resources. Users can rent just the right amount of computing power for this specific task without overpaying for unused capacity, making it an economical choice for audio processing workflows that might otherwise be cost-prohibitive on traditional cloud platforms.


## Choosing an Instance

For running the Pyannote Speaker Diarization model on VAST.ai, you'll need a relatively modest GPU setup. The pyannote/speaker-diarization-3.1 model runs in pure PyTorch and is designed to be efficient. Here are the recommended specifications:

- GPU: A low-end GPU like an RTX 3060 or 4060 would be sufficient.
- VRAM: 6-8GB of VRAM should be adequate as the Pyannote diarization pipeline is relatively efficient.
- RAM: 8-16GB system RAM is recommended for processing audio files.
- Storage: At least 10GB for the model, dependencies, and your audio files.
- CUDA: Make sure the instance has CUDA installed (version 11.0+ recommended).
- Python: Python 3.8+ with PyTorch installed.


## Renting an Instance on Vast.ai

1. Ensure that you have a Vast.ai account
2. Go to the Vast Templates in the Console https://cloud.vast.ai/templates/
3. Select the `PyTorch (CuDNN Runtime)` Template
4. Filter for an instance with:
- 1 GPU
- 6-8GB of VRAM     
- 8-16GB system RAM
- 10GB of storage
5. Select an instance and click rent
6. Install the Vast TLS certificate in your browser to access the notebook server https://docs.vast.ai/instances/jupyter#1SmCz
7. Go to your Instances https://cloud.vast.ai/instances/ and click "Open" to access the jupyter server on your instance.
8. Upload this notebook to the server

## Install Dependencies

In [ ]:
%%bash
pip install pyannote.audio
pip install pydub
pip install librosa
pip install datasets

In [ ]:
%%bash
apt-get update && apt-get install -y ffmpeg

## Set up your Huggingface Token

Here we set our huggingface token as `HF_TOKEN`. We need this to access the model.

Ensure that you have accepted the terms for https://huggingface.co/pyannote/speaker-diarization-3.1 and https://huggingface.co/pyannote/segmentation-3.0. This model is free to use, but you must accept their terms.

In [1]:
# Make sure you've accepted the user conditions at:
# https://huggingface.co/pyannote/speaker-diarization-3.1
# https://huggingface.co/pyannote/segmentation-3.0

HF_TOKEN = ""

## Download Test Data

We will use a sample file from the AMI Meeting Corpus dataset https://huggingface.co/datasets/diarizers-community/ami, which is a collection of 100 hours of meeting recordings.

This code efficiently pulls a few sample files from the dataset. If you want to download the entire dataset there are better methods - see the Huggingface API.

In [ ]:
from datasets import load_dataset
import os
import soundfile as sf

# Create a directory to save the files
os.makedirs("ami_samples", exist_ok=True)

# Load the dataset with the correct split
dataset = load_dataset("diarizers-community/ami", "ihm", split="train", streaming=True)


# load any number of samples
n_samples = 1
samples = list(dataset.take(n_samples))

for i, sample in enumerate(samples):

    audio = sample["audio"]
    audio_array = audio["array"]
    sampling_rate = audio["sampling_rate"]
    
    # Calculate duration in seconds
    duration = len(audio_array) / sampling_rate
    
    # Use soundfile to save the audio
    output_path = f"ami_samples/sample_{i}.wav"
    sf.write(output_path, audio_array, sampling_rate)
    
    print(f"Saved {output_path} - Speaker: {sample['speakers']} - Duration: {duration:.2f} seconds")

## Speaker Diarization

First we set up the Speaker Diarization pipeline.

In [ ]:

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from pyannote.audio import Pipeline

pipeline = Pipeline.from_pretrained(
        "pyannote/speaker-diarization-3.1",
        use_auth_token=HF_TOKEN
)

# Move pipeline to appropriate device
pipeline = pipeline.to(device)

### Get Diarization Results
Next, we process the file to get the timestamps where speech starts and ends.

In [ ]:
# Process the audio file
audio_file = "./ami_samples/sample_0.wav"
print(f"Processing {audio_file} on {device}")
output = pipeline(audio_file)

The Pyannote Speaker Diarization model gives us a list of segment timestamps labeled with a speaker. 

In [ ]:
print("Voice activity segments:")

for segment, _, speaker in output.itertracks(yield_label=True):
        result = f"{segment.start:.2f} --> {segment.end:.2f} (duration: {segment.duration:.2f}s) Speaker: {speaker}"
        print(result)

### Additional Analytics

Now that we have processed our file we'll explore a few useful features of the Pyannote SDK:

1. See total speaker time broken down by speaker.
2. Find segments with speaker overlap (multiple speakers speaking at once).
3. Filter the data by speaker.

#### Speaker Time

Here we see the total speaking time for each speaker.

In [ ]:
for speaker in output.labels():
    speaking_time = output.label_duration(speaker)
    print(f"Speaker {speaker} total speaking time: {speaking_time:.2f}s")

#### Speaker Overlap

Pyannote shows us the timestamps where multiple speakers are speaking.

In [ ]:
overlap = output.get_overlap()
print(f"Overlapping speech regions: {overlap}")

#### Filter the Data by Speaker

We can use Pyannote to filter the output by speaker.

In [ ]:
speaker = "SPEAKER_06"
speaker_turns = output.label_timeline(speaker)
print(f"Speaker {speaker} speaks at:")
for speaker_turn in speaker_turns:
    print(speaker_turn)

## Inspect results

Next, we'll split the audio into chunks based on the diarization output in order to verify that it successfully isolated the speakers.


### Split the Audio

Here we write a function to split the original audio into segments determined by our Diarization output.

In [12]:
import shutil
from pydub import AudioSegment

def split_audio_by_segments(audio_path, diarization_output, output_dir="output_segments"):
    """
    Split an audio file into multiple files based on diarization output
    
    Parameters:
    -----------
    audio_path: str
        Path to the input audio file
    diarization_output: Annotation
        Pyannote diarization output
    output_dir: str
        Directory to save the output segments
    """
    # Clear the output directory if it exists
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Load the audio file
    audio = AudioSegment.from_file(audio_path)
    
    # Extract each segment with speaker information
    for i, (segment, _, speaker) in enumerate(diarization_output.itertracks(yield_label=True)):
        # Convert seconds to milliseconds
        start_ms = int(segment.start * 1000)
        end_ms = int(segment.end * 1000)
        
        # Extract segment
        segment_audio = audio[start_ms:end_ms]
        
        # Generate output filename with speaker information
        filename = os.path.basename(audio_path)
        name, ext = os.path.splitext(filename)
        output_path = os.path.join(output_dir, f"{name}_segment_{i+1:04d}_{start_ms:08d}ms-{end_ms:08d}ms_speaker_{speaker}{ext}")
        
        # Export segment
        segment_audio.export(output_path, format=ext.replace('.', ''))
        print(f"Saved segment {i+1} to {output_path} (Speaker: {speaker})")

We then use that to save the segments to a local folder

In [ ]:
split_audio_by_segments(audio_file, output)

### Inspect Results

Here we create a function that allows us to play the audio files in our notebook.

In [14]:
import librosa
from IPython.display import Audio, display

def play_audio(file_path, sr=None):
    """
    Play an audio file in a Jupyter notebook.
    
    Parameters:
    -----------
    file_path : str
        Path to the audio file to play
    sr : int, optional
        Sample rate to load the audio with. If None, uses the file's native sample rate.
        
    Returns:
    --------
    Audio widget that can be played in the notebook
    
    Example:
    --------
    >>> play_audio('path/to/audio.wav')
    """
    # Load the audio file
    y, sr = librosa.load(file_path, sr=sr)
    
    
    # Display an audio widget to play the sound
    audio_widget = Audio(data=y, rate=sr)
    display(audio_widget)

We'll use `play_audio` to listen to a few clips to verify that the speakers were correctly identified and isolated.

In [ ]:
import os
audio_dir = "./output_segments/"

audio_files = os.listdir(audio_dir)
audio_files.sort()

n_offset = 21
n_clips = 5

for fname in audio_files[n_offset:n_clips + n_offset]:
    print(f"File: {fname}")
    
    # Extract speaker info if present in filename
    if "_speaker_" in fname:
        speaker_part = fname.split("_speaker_")[1].split(".")[0]
        print(f"Speaker: {speaker_part}")
    
    play_audio(audio_dir + fname)

### Verify Speaker Overlap

Sometimes we find clips with multiple speakers speaking. We can check the overlap file to verify that there are multiple speakers speaking at that time. 

In [ ]:
overlap = output.get_overlap()
for overlap_ts in overlap:
    print(f"Overlapping speech regions: {overlap_ts}")

## Conclusion

The Pyannote speaker diarization model successfully identified multiple distinct speakers in the AMI Meeting Corpus sample. The model accurately detected overlapping speech regions, which we confirmed through our audio extraction and playback tests, demonstrating its effectiveness at handling complex conversational dynamics.
